# Intent labeling — MongoDB + gpt-4o-mini + verifier + guardrail

Pipeline 2 bước: **truy hồi ứng viên từ MongoDB** → **Annotate (gpt-4o-mini)** → **Verify (gpt-4o, §6.5)** → **guardrail / QA status** → lưu `intent_annotations`.

Xem `LABELING_GUIDE.md` (auto-add nhãn mới mặc định **confidence ≥ 0.96**).

**Verifier:** model mạnh hơn (`VERIFIER_MODEL`, mặc định `gpt-4o`) đọc lại nhãn annotator → `RETAIN` hoặc `REVISE`. Mặc định chỉ verify khi `confidence < VERIFY_CONF_THRESHOLD` (0.85); đặt `VERIFY_ALL_SAMPLES=1` để verify mọi mẫu như diagram.

**Human review:** các mẫu `needs_review` và `pending_new_label_review` được người rà soát sau pipeline.

**Điều kiện:** graph đã có trong MongoDB (`intent_nodes` / `intent_edges`) — build từ `data/intent_graph_rag_colab.ipynb`.


## 1. Cài đặt

> **LLM**: **OpenAI `gpt-4o-mini`** qua API — chỉ cần `OPENAI_API_KEY`, không cần GPU.


In [1]:
import sys
print(sys.executable)


/venv/main/bin/python


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
pip install -q pymongo certifi sentence-transformers openai

Note: you may need to restart the kernel to use updated packages.


## 1.5. Mount Google Drive (Colab)

Khi chạy trên **Google Colab**, ô bên dưới sẽ:

1. Mount Google Drive vào `/content/drive`.
2. Set `INTENT_REPO=/content/drive/MyDrive/intent_kb` (cấu trúc: `intent_kb/data/hasaki_prelabel.json`, `intent_kb/data/outputs/...`).
3. Cell §2 sẽ tự đọc `INTENT_REPO` này nên các đường dẫn `HASAKI_JSON`, `OUTPUT_DIR` đều trỏ thẳng vào Drive ⇒ checkpoint **không mất** khi runtime disconnect.

Khi chạy local thì cell tự bỏ qua. Có thể override bằng env `INTENT_REPO` trước khi mở notebook.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Mặc định cho Colab: My Drive/intent_kb (đổi nếu folder của bạn tên khác)
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

    os.environ.setdefault("INTENT_REPO", DEFAULT_DRIVE_REPO)
    repo = Path(os.environ["INTENT_REPO"])
    print(f"📁 INTENT_REPO = {repo}")
    if not repo.exists():
        raise FileNotFoundError(
            f"Không thấy {repo} trong Drive. "
            f"Kiểm tra lại tên folder trong My Drive (mong đợi: 'intent_kb')."
        )
    expected = repo / "data" / "hasaki_prelabel.json"
    print(f"   hasaki_prelabel.json: {'✅' if expected.exists() else '❌'} {expected}")
else:
    print("ℹ️ Không phải Colab — bỏ qua mount Drive. INTENT_REPO sẽ lấy từ env hoặc '.'")

## 2. Cấu hình


In [ ]:
import os
import getpass
from pathlib import Path

REPO_ROOT = Path(os.environ.get("INTENT_REPO", ".")).resolve()
HASAKI_JSON = REPO_ROOT / "data/hasaki_prelabel.json"
# Output (checkpoint + file labelled cuoi). Tren Colab REPO_ROOT da tro vao Drive,
# nen OUTPUT_DIR mac dinh cung nam tren Drive -> khong mat khi runtime disconnect.
# Co the override bang env INTENT_OUTPUT_DIR (vd. /content/drive/MyDrive/khac/outputs).
_out_env = (os.environ.get("INTENT_OUTPUT_DIR") or "").strip()
OUTPUT_DIR = Path(_out_env).resolve() if _out_env else (REPO_ROOT / "data" / "outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_NAME = os.environ.get("MONGODB_DB", "intent_kb")
COL_NODES = "intent_nodes"
COL_EDGES = "intent_edges"
# Khong hardcode URI: uu tien bien moi truong MONGODB_URI, neu trong thi nhap khi chay cell (an khi go)
MONGODB_URI = (os.environ.get("MONGODB_URI") or "").strip()
if not MONGODB_URI:
    MONGODB_URI = getpass.getpass("MONGODB_URI (dan vao roi Enter): ").strip()

# === LLM: OpenAI gpt-4o-mini (API) ===
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_TEMPERATURE = float(os.environ.get("OPENAI_TEMPERATURE", "0"))
OPENAI_MAX_RETRIES = int(os.environ.get("OPENAI_MAX_RETRIES", "3"))
# OPENAI_API_KEY: nhập ở cell §2.5 ngay dưới (getpass) — không lưu vào notebook.
OPENAI_API_KEY = ""

MIN_CONF_APPROVE_EXISTING = 0.85
MIN_CONF_AUTO_ADD_NEW_LABEL = 0.80
ALLOW_AUTO_ADD_NEW_LABEL = True

# === Verifier (cross-verification, §6.5) ===
# Annotator: gpt-4o-mini | Verifier: gpt-4o (model manh hon, doc lai nhan).
ENABLE_VERIFIER = bool(int(os.environ.get("ENABLE_VERIFIER", "1")))
VERIFIER_MODEL = os.environ.get("VERIFIER_MODEL", "gpt-4o")
# VERIFY_ALL_SAMPLES=1: verify moi mau (khop diagram 2-step). Mac dinh 0: chi verify khi conf thap.
VERIFY_ALL_SAMPLES = bool(int(os.environ.get("VERIFY_ALL_SAMPLES", "0")))
# Chi goi verifier khi confidence < nguong nay (bo qua neu VERIFY_ALL_SAMPLES=1).
VERIFY_CONF_THRESHOLD = float(os.environ.get("VERIFY_CONF_THRESHOLD", "0.85"))
VERIFY_MAX_TOKENS = int(os.environ.get("VERIFY_MAX_TOKENS", "256"))

BATCH_START = 0
HASAKI_BATCH_SIZE = 200
# Resume chay full prelabel: index bat dau (0 = tu dau file)
HASAKI_RUN_OFFSET = 0
# Sau khi export JSON: xoa BATCH_OUTPUT khoi RAM (checkpoint van tren dia)
HASAKI_FREE_MEM_AFTER_EXPORT = True
BATCH_LIMIT = HASAKI_BATCH_SIZE
TOP_K_CANDIDATES = 5
MAX_NEW_TOKENS = 384

# Human review target statuses (xuất ra để người rà soát sau pipeline).
_human_review_env = os.environ.get(
    "HUMAN_REVIEW_QA_STATUSES", "needs_review,pending_new_label_review"
)
HUMAN_REVIEW_QA_STATUSES = {
    s.strip() for s in _human_review_env.split(",") if s.strip()
}

print("HASAKI_JSON:", HASAKI_JSON.exists(), HASAKI_JSON)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("MONGODB_URI set:", bool(MONGODB_URI))
print("LLM model:", OPENAI_MODEL)
print("Verifier:", "ON" if ENABLE_VERIFIER else "OFF", "| model:", VERIFIER_MODEL,
      "| verify all:", VERIFY_ALL_SAMPLES,
      "| verify if conf <", VERIFY_CONF_THRESHOLD if not VERIFY_ALL_SAMPLES else "N/A")
print("Human review statuses:", sorted(HUMAN_REVIEW_QA_STATUSES))


HASAKI_JSON: True /workspace/data/hasaki_prelabel.json
MONGODB_URI set: True


## 2.5. Nhập OPENAI_API_KEY

Chạy ô bên dưới để nhập key (ẩn khi gõ qua `getpass`). Có thể chạy lại bất cứ lúc nào để **đổi key** mà không phải động vào §2.

- Nếu env `OPENAI_API_KEY` đã được set sẵn (Colab Secret / shell `export`) → tự động dùng và **bỏ qua** prompt.


In [ ]:
env_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
if env_key:
    OPENAI_API_KEY = env_key
    print("✅ OPENAI_API_KEY lấy từ biến môi trường.")
else:
    OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY (dán vào rồi Enter): ").strip()

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY rỗng — chạy lại cell này và dán key, "
        "hoặc set env OPENAI_API_KEY trước khi mở notebook."
    )
if not OPENAI_API_KEY.startswith(("sk-", "sk_")):
    print("⚠️ Key không bắt đầu bằng 'sk-' — kiểm tra lại nếu gọi API lỗi 401.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"✅ OPENAI_API_KEY set (len={len(OPENAI_API_KEY)}, suffix=…{OPENAI_API_KEY[-4:]})")


## 3. Kết nối MongoDB


In [5]:
import certifi
from pymongo import MongoClient

if not MONGODB_URI:
    raise ValueError(
        "Chua co MONGODB_URI. Dat bien moi truong MONGODB_URI hoac chay lai cell cau hinh de nhap."
    )

client = MongoClient(
    MONGODB_URI,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=15000,
)
client.admin.command("ping")
db = client[DB_NAME]
print("OK db:", db.name)
print("intent_nodes:", db[COL_NODES].count_documents({}))
print("intent_edges:", db[COL_EDGES].count_documents({}))


OK db: intent_kb
intent_nodes: 181
intent_edges: 182


## 3.5. Migration L1: `truoc ban` / `sau ban` → `truoc_mua_hang` / `sau_mua_hang`

Chạy **một lần** sau khi đổi taxonomy. Cell này:

1. Update tất cả documents trong `intent_annotations` có `intent.level_1` là slug cũ → slug mới.
2. (Tùy chọn) update các `intent_nodes` còn sót L1 cũ nếu graph chưa rebuild qua `intent_graph_rag_colab.ipynb`.
3. In ra số document đã thay đổi và list các giá trị L1 hiện có sau migration.

Set `DRY_RUN = True` để chỉ đếm, không thực sự update. Sau khi xem xét OK thì đổi `False` và chạy lại.

> ⚠️ Nếu đã chạy lại §5 trong `intent_graph_rag_colab.ipynb` (drop & rebuild graph từ CSV mới) thì phần graph đã sạch — chỉ cần phần `intent_annotations`.


In [ ]:
DRY_RUN = True  # đổi sang False để thực sự update

L1_MAP = {
    "truoc ban": "truoc_mua_hang",
    "truoc_ban": "truoc_mua_hang",
    "trước bán": "truoc_mua_hang",
    "trước ban": "truoc_mua_hang",
    "truoc bán": "truoc_mua_hang",
    "sau ban": "sau_mua_hang",
    "sau_ban": "sau_mua_hang",
    "sau bán": "sau_mua_hang",
}

print(f"=== Migration L1 slugs ({'DRY-RUN' if DRY_RUN else 'APPLY'}) ===\n")

# 1) intent_annotations.intent.level_1
ann_col = db["intent_annotations"]
ann_before = {old: ann_col.count_documents({"intent.level_1": old}) for old in L1_MAP}
ann_before = {k: v for k, v in ann_before.items() if v > 0}
print(f"intent_annotations cần migrate: {ann_before} (tổng {sum(ann_before.values())})")

if not DRY_RUN and ann_before:
    total = 0
    for old, new in L1_MAP.items():
        res = ann_col.update_many(
            {"intent.level_1": old},
            {"$set": {"intent.level_1": new}},
        )
        if res.modified_count:
            print(f"  ✏️  {old!r} -> {new!r}: {res.modified_count} docs")
            total += res.modified_count
    print(f"  ✅ Tổng cập nhật intent_annotations: {total}")

# 2) intent_nodes L1 (chỉ áp dụng nếu graph chưa rebuild)
node_col = db[COL_NODES]
node_l1_before = {old: node_col.count_documents({"l1": old}) for old in L1_MAP}
node_l1_before = {k: v for k, v in node_l1_before.items() if v > 0}
print(f"\nintent_nodes (level=intent) còn L1 cũ: {node_l1_before}")

stale_l1_nodes = list(node_col.find({"level": "l1", "_id": {"$in": [f"l1:{old}" for old in L1_MAP]}}, {"_id": 1, "name": 1}))
if stale_l1_nodes:
    print(f"  ⚠️ Còn {len(stale_l1_nodes)} L1 nodes cũ trong graph: {[n['_id'] for n in stale_l1_nodes]}")
    print(f"  👉 Khuyến nghị: chạy lại §5 trong `intent_graph_rag_colab.ipynb` (drop & rebuild) thay vì patch chỗ này.")

if not DRY_RUN and node_l1_before:
    total = 0
    for old, new in L1_MAP.items():
        res = node_col.update_many({"l1": old}, {"$set": {"l1": new}})
        if res.modified_count:
            print(f"  ✏️  intent_nodes.l1 {old!r} -> {new!r}: {res.modified_count} docs")
            total += res.modified_count
    print(f"  ✅ Tổng cập nhật intent_nodes.l1: {total}")

# 3) Báo cáo phân phối L1 hiện tại
print("\n=== Distribution L1 sau migration ===")
print("intent_annotations.intent.level_1:")
for doc in ann_col.aggregate([{"$group": {"_id": "$intent.level_1", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}]):
    print(f"  {doc['_id']!r:30s}: {doc['n']}")
print("intent_nodes (level=intent) .l1:")
for doc in node_col.aggregate([{"$match": {"level": "intent"}}, {"$group": {"_id": "$l1", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}]):
    print(f"  {doc['_id']!r:30s}: {doc['n']}")

## 4. Helper: upsert graph, guardrail, retrieval


In [ ]:
from datetime import datetime, timezone
import json
import re
import unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ngưỡng snap nhãn intent có sẵn qua cosine vs embedding đã enrich (đối xứng với retrieval)
SEMANTIC_SNAP_TO_INTENT_THRESHOLD = 0.85


# Global semantic encoder - lazy loaded
_semantic_model = None

def get_semantic_model():
    """Lazy load sentence transformer model"""
    global _semantic_model
    if _semantic_model is None:
        print("Loading paraphrase-multilingual-MiniLM-L12-v2...")
        _semantic_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _semantic_model


def build_graph_aware_intent_text(intent_doc, db):
    """Build graph-aware description for intent node embedding
    
    Args:
        intent_doc: intent node document from MongoDB
        db: MongoDB database connection
    
    Returns:
        str: graph-aware text for embedding
    """
    # Extract path components
    l1 = intent_doc.get('l1', '')
    l2 = intent_doc.get('l2', '') 
    l3 = intent_doc.get('l3', '')
    description = intent_doc.get('description', '')
    detection_signals = intent_doc.get('detection_signals', '')
    example = intent_doc.get('example', '')
    
    # Build full path
    full_path = f"{l1}.{l2}.{l3}"
    
    # Get sibling intents (same L2, limit 5)
    siblings_cursor = db[COL_NODES].find(
        {'level': 'intent', 'l2': l2, '_id': {'$ne': intent_doc['_id']}}, 
        {'l3': 1}
    ).limit(5)
    siblings = [doc.get('l3', '') for doc in siblings_cursor if doc.get('l3')]
    siblings_text = ', '.join(siblings) if siblings else 'None'
    
    # Construct graph-aware text
    graph_text = f"""Intent: {full_path}.
Parent: {l2}.
Siblings: {siblings_text}.
Description: {description}
Signals: {detection_signals}
Example: {example}"""
    
    return graph_text


def embed_and_store_intent_nodes(db, *, force=False):
    """Embedding enrichment cho toan bo intent nodes.

    - force=False (mac dinh): bo qua node da co `embedding` + `graph_aware_text`.
    - force=True: re-embed TAT CA (dung sau khi rebuild graph tu CSV moi,
      doi sentence-transformer model, hoac sua build_graph_aware_intent_text).
    """
    model = get_semantic_model()
    col_nodes = db[COL_NODES]

    intent_nodes = list(col_nodes.find({'level': 'intent'}))
    print(f"Found {len(intent_nodes)} intent nodes to process (force={force})")

    updated_count = 0
    skipped_count = 0
    for intent_doc in intent_nodes:
        if not force and 'embedding' in intent_doc and 'graph_aware_text' in intent_doc:
            skipped_count += 1
            continue

        graph_text = build_graph_aware_intent_text(intent_doc, db)

        # Generate embedding
        embedding = model.encode([graph_text])[0]
        # Normalize embedding
        embedding = embedding / np.linalg.norm(embedding)
        
        # Store in MongoDB
        col_nodes.update_one(
            {'_id': intent_doc['_id']},
            {
                '$set': {
                    'graph_aware_text': graph_text,
                    'embedding': embedding.tolist(),  # Convert to list for MongoDB
                    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
                    'embedding_updated_at': datetime.now(timezone.utc)
                }
            }
        )
        updated_count += 1

        if updated_count % 20 == 0:
            print(f"Processed {updated_count} nodes...")

    print(f"✅ Embedded {updated_count} | ⏭️ Skipped {skipped_count} (đã có embedding) | Total {len(intent_nodes)}")
    if skipped_count == len(intent_nodes) and not force:
        print("   ⚠️ Tất cả node đã có embedding cũ. Sau khi đổi taxonomy/CSV → gọi `embed_and_store_intent_nodes(db, force=True)` để rebuild.")
    return updated_count


def semantic_retrieval(db, query_text, top_k=10, min_similarity=0.3):
    """Semantic retrieval using graph-aware embeddings
    
    Args:
        db: MongoDB database connection
        query_text: user query sentence
        top_k: number of candidates to return
        min_similarity: minimum cosine similarity threshold
    
    Returns:
        list: intent documents with similarity scores
    """
    model = get_semantic_model()
    col_nodes = db[COL_NODES]
    
    # Encode query
    query_embedding = model.encode([query_text])[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    
    # Find intent nodes with embeddings
    intent_nodes = list(col_nodes.find(
        {'level': 'intent', 'embedding': {'$exists': True}},
        {
            '_id': 1, 'name': 1, 'l1': 1, 'l2': 1, 'l3': 1,
            'description': 1, 'detection_signals': 1, 'example': 1,
            'product_category': 1, 'embedding': 1
        }
    ))
    
    if not intent_nodes:
        print("Warning: No intent nodes with embeddings found")
        return []
    
    # Calculate similarities
    scored_candidates = []
    for intent_doc in intent_nodes:
        stored_embedding = np.array(intent_doc['embedding'])
        
        # Compute cosine similarity
        similarity = cosine_similarity([query_embedding], [stored_embedding])[0][0]
        
        if similarity >= min_similarity:
            # Add similarity score to document
            intent_doc['semantic_score'] = float(similarity)
            scored_candidates.append(intent_doc)
    
    # Sort by similarity score descending
    scored_candidates.sort(key=lambda x: x['semantic_score'], reverse=True)
    
    return scored_candidates[:top_k]


def union_retrieval(db, text, category=None, top_k_regex=8, top_k_semantic=6, sample=None):
    """UNION of regex and semantic retrieval with deduplication
    
    Args:
        db: MongoDB database connection
        text: query text
        category: product category (optional)
        top_k_regex: max regex candidates
        top_k_semantic: max semantic candidates
        sample: sample data for context matching
    
    Returns:
        list: deduplicated union of candidates
    """
    # Get regex candidates (existing logic)
    regex_candidates = get_candidate_l3_from_mongodb(db, text, category, top_k_regex, sample)
    
    # Get semantic candidates (new logic)
    semantic_candidates = semantic_retrieval(db, text, top_k_semantic)
    
    # Deduplicate by _id, preserving document structure
    seen_ids = set()
    union_candidates = []
    
    # Add regex candidates first (preserve original scoring)
    for candidate in regex_candidates:
        if candidate['_id'] not in seen_ids:
            candidate['retrieval_method'] = 'regex'
            union_candidates.append(candidate)
            seen_ids.add(candidate['_id'])
    
    # Add semantic candidates
    for candidate in semantic_candidates:
        if candidate['_id'] not in seen_ids:
            candidate['retrieval_method'] = 'semantic'
            union_candidates.append(candidate)
            seen_ids.add(candidate['_id'])
    
    # Limit total candidates
    max_total = max(top_k_regex + top_k_semantic, TOP_K_CANDIDATES)
    return union_candidates[:max_total]


def find_nearest_existing_intent(db, pred_text, threshold=SEMANTIC_SNAP_TO_INTENT_THRESHOLD):
    """Embed prediction path/slug; cosine vs intent node embeddings. Returns (node, sim) or (None, best_sim)."""
    text = (pred_text or "").strip()
    if not text:
        return None, -1.0
    model = get_semantic_model()
    q = np.asarray(model.encode([text])[0], dtype=np.float64)
    n = np.linalg.norm(q)
    if n < 1e-9:
        return None, -1.0
    q = q / n
    nodes = list(
        db[COL_NODES].find(
            {"level": "intent", "embedding": {"$exists": True}},
            {"_id": 1, "l1": 1, "l2": 1, "l3": 1, "embedding": 1},
        )
    )
    best, best_sim = None, -1.0
    for intent_doc in nodes:
        ev = np.asarray(intent_doc["embedding"], dtype=np.float64)
        en = np.linalg.norm(ev)
        if en < 1e-9:
            continue
        sim = float(np.dot(q, ev / en))
        if sim > best_sim:
            best_sim, best = sim, intent_doc
    if best is not None and best_sim >= threshold:
        return best, best_sim
    return None, best_sim


def upsert_new_label_to_graph(
    db,
    *,
    domain,
    l1,
    l2,
    l3,
    intent_id,
    intent_name,
    description="",
    confidence="",
    product_category="",
    detection_signals="",
    example="",
    category_logic="",
):
    col_nodes = db[COL_NODES]
    col_edges = db[COL_EDGES]
    domain_id = f"domain:{domain}"
    l1_id, l2_id, l3_id = f"l1:{l1}", f"l2:{l2}", f"l3:{l3}"
    now = datetime.now(timezone.utc)

    for node_id, level, name, extra in [
        ("ROOT", "root", "Intent Knowledge Base", {}),
        (domain_id, "domain", domain, {}),
        (l1_id, "l1", l1, {}),
        (l2_id, "l2", l2, {}),
        (l3_id, "l3", l3, {}),
        (
            intent_id,
            "intent",
            intent_name,
            {
                "description": description,
                "confidence": confidence,
                "domain": domain,
                "product_category": product_category,
                "l1": l1,
                "l2": l2,
                "l3": l3,
                "detection_signals": detection_signals,
                "example": example,
                "category_logic": category_logic,
            },
        ),
    ]:
        col_nodes.update_one(
            {"_id": node_id},
            {"$setOnInsert": {"_id": node_id, "level": level}, "$set": {"name": name, **extra, "updated_at": now}},
            upsert=True,
        )

    for src, dst in [
        ("ROOT", domain_id),
        (domain_id, l1_id),
        (l1_id, l2_id),
        (l2_id, l3_id),
        (l3_id, intent_id),
    ]:
        eid = f"{src}->{dst}"
        col_edges.update_one(
            {"_id": eid},
            {"$setOnInsert": {"_id": eid, "from": src, "to": dst}, "$set": {"updated_at": now}},
            upsert=True,
        )
    return {"intent_id": intent_id, "status": "upserted"}


def _norm_label(s):
    s = unicodedata.normalize("NFC", (s or "").strip())
    return re.sub(r"\s+", " ", s).lower()


def _collect_allowed_taxonomy(db):
    l1_set, l2_set, l3_set = set(), set(), set()
    for doc in db[COL_NODES].find({"level": "l1"}, {"name": 1}):
        if doc.get("name"):
            l1_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l2"}, {"name": 1}):
        if doc.get("name"):
            l2_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l3"}, {"name": 1}):
        if doc.get("name"):
            l3_set.add(doc["name"])

    # norm -> chuỗi canonical trong Mongo (tranh trùng key: lấy deterministic theo sort)
    def _norm_to_canon(name_set):
        m = {}
        for name in sorted(name_set):
            k = _norm_label(name)
            if k and k not in m:
                m[k] = name
        return m

    return {
        "l1": l1_set,
        "l2": l2_set,
        "l3": l3_set,
        "norm_canon_l1": _norm_to_canon(l1_set),
        "norm_canon_l2": _norm_to_canon(l2_set),
        "norm_canon_l3": _norm_to_canon(l3_set),
    }


def _snap_pred_to_canonical_taxonomy(pred, allowed):
    """Ghi đè L1/L2/L3 bằng tên đúng trong graph nếu normalize khớp một nhãn có sẵn."""
    for lvl, key in [("l1", "L1"), ("l2", "L2"), ("l3", "L3")]:
        raw = pred.get(key)
        if raw is None:
            continue
        raw = str(raw).strip()
        if not raw:
            continue
        cmap = allowed.get(f"norm_canon_{lvl}", {})
        canon = cmap.get(_norm_label(raw))
        if canon:
            pred[key] = canon
    # KHÔNG ép L1 sang 'truoc_mua_hang' khi model trả slug domain/ngành làm L1
    # ('my pham', 'trang diem', 'gia_so_sanh', 'san_pham_so_sanh', ...).
    # Để pred giữ nguyên để _validate_pred_for_auto_create chặn upsert
    # và rơi vào pending_new_label_invalid_slug cho human review.


def _is_label_allowed(pred, allowed):
    n1, n2, n3 = (
        _norm_label(pred.get("L1")),
        _norm_label(pred.get("L2")),
        _norm_label(pred.get("L3")),
    )
    nl1 = {_norm_label(x) for x in allowed["l1"]}
    nl2 = {_norm_label(x) for x in allowed["l2"]}
    nl3 = {_norm_label(x) for x in allowed["l3"]}
    return (n1 in nl1) and (n2 in nl2) and (n3 in nl3)


# === Guardrail: validate L1/L2/L3 trước khi tạo intent mới trong graph ===
# Chỉ cho phép L1 ∈ {truoc_mua_hang, sau_mua_hang}; L2/L3 phải là slug ASCII lowercase + underscore.
# Reject các trường hợp model nhầm slug L2 (gia_so_sanh, ...) hoặc domain (My pham, Trang diem) vào L1.

_ALLOWED_L1_NORM_TO_CANON = {
    "truoc_mua_hang": "truoc_mua_hang",
    "truoc mua hang": "truoc_mua_hang",
    "truoc_ban": "truoc_mua_hang",
    "truoc ban": "truoc_mua_hang",
    "trước mua hàng": "truoc_mua_hang",
    "trước bán": "truoc_mua_hang",
    "trước ban": "truoc_mua_hang",
    "truoc bán": "truoc_mua_hang",
    "sau_mua_hang": "sau_mua_hang",
    "sau mua hang": "sau_mua_hang",
    "sau_ban": "sau_mua_hang",
    "sau ban": "sau_mua_hang",
    "sau mua hàng": "sau_mua_hang",
    "sau bán": "sau_mua_hang",
}

_L2_SLUGS_OFTEN_CONFUSED_AS_L1 = {
    "gia_so_sanh", "san_pham_so_sanh", "thiet_bi_tu_van", "chat_luong_loi",
    "di_ung_an_toan", "ton_kho_kiem_tra", "uu_dai_yeu_cau", "khieu_nai_dich_vu",
    "trang_thai_van_chuyen", "thay_doi_don_hang", "doi_tra", "hoan_tien_yeu_cau",
    "hong_vo_san_pham", "giao_hang_loi", "van_de_giao_hang", "thong_tin_san_pham",
    "tinh_nang_truy_van", "thong_sos_truy_van",
}

_SLUG_RE = re.compile(r"^[a-z][a-z0-9_]+$")


def _validate_pred_for_auto_create(pred):
    """Kiểm tra (L1, L2, L3) trước khi upsert intent mới vào graph.

    Trả về (valid: bool, reason: str, normalized: dict).
    Quy tắc:
    - L1 phải normalize được về 'truoc_mua_hang' hoặc 'sau_mua_hang'
    - L2 / L3: slug ASCII lowercase + underscore, không trùng nhau, không rỗng
    - L2 không được là một trong các slug L2 phổ biến mà model hay nhầm lên L1
    """
    out = {"L1": "", "L2": "", "L3": ""}
    l1_raw = (pred.get("L1") or "").strip()
    l2_raw = (pred.get("L2") or "").strip()
    l3_raw = (pred.get("L3") or "").strip()

    if not (l1_raw and l2_raw and l3_raw):
        return False, "empty_l1_l2_l3", out

    l1_canon = _ALLOWED_L1_NORM_TO_CANON.get(_norm_label(l1_raw))
    if not l1_canon:
        return False, f"l1_must_be_truoc_or_sau_mua_hang_got:{l1_raw!r}", out
    out["L1"] = l1_canon

    if not _SLUG_RE.match(l2_raw):
        return False, f"l2_invalid_slug:{l2_raw!r}", out
    if l2_raw in _ALLOWED_L1_NORM_TO_CANON:
        return False, f"l2_looks_like_l1:{l2_raw!r}", out
    out["L2"] = l2_raw

    if not _SLUG_RE.match(l3_raw):
        return False, f"l3_invalid_slug:{l3_raw!r}", out
    if l3_raw == l2_raw:
        return False, "l3_eq_l2", out
    if l3_raw in _L2_SLUGS_OFTEN_CONFUSED_AS_L1 or l3_raw in _ALLOWED_L1_NORM_TO_CANON:
        return False, f"l3_looks_like_higher_level:{l3_raw!r}", out
    out["L3"] = l3_raw

    return True, "", out


def is_sentence_too_ambiguous_to_annotate(sentence, min_length=6, min_meaningful_words=1):
    """Check if sentence is too short/ambiguous to reliably annotate
    
    Args:
        sentence: input sentence text
        min_length: minimum character length (excluding spaces)
        min_meaningful_words: minimum meaningful words (length >= 2)
    
    Returns:
        tuple: (is_ambiguous: bool, reason: str)
    """
    if not sentence or not sentence.strip():
        return True, "empty_sentence"
    
    text = sentence.strip()
    
    # Check character length (excluding spaces) - reduced from 8 to 6
    char_count = len(re.sub(r'\s+', '', text))
    if char_count < min_length:
        return True, f"too_short_chars_{char_count}"
    
    # Extract meaningful words (length >= 2, more lenient)
    words = re.findall(r'\w+', text.lower(), flags=re.UNICODE)
    # Reduced stop words list - keep common meaningful words
    stop_words = {'có', 'thế', 'này', 'kia', 'được', 'như'}
    meaningful_words = [w for w in words if len(w) >= 2 and w not in stop_words]
    
    # Special handling for price/money questions  
    money_words = {'tiền', 'giá', 'bao', 'nhiều', 'cost', 'price'}
    product_words = {'sản', 'phẩm', 'sp', 'hàng', 'tốt', 'xấu', 'quality'}
    
    has_money_context = any(w in ' '.join(meaningful_words) for w in money_words)
    has_product_context = any(w in ' '.join(meaningful_words) for w in product_words)
    
    # Allow price questions even if short
    if has_money_context or has_product_context:
        min_meaningful_words = 1
    
    if len(meaningful_words) < min_meaningful_words:
        return True, f"too_few_meaningful_words_{len(meaningful_words)}"
    
    # Check if sentence is just question words without ANY context
    question_only_patterns = [
        r'^(có\s*)?(không\s*)?(gì\s*\?*\s*)$',  # "Gì?"
        r'^(sao\s*\?*\s*)$',                    # "Sao"
        r'^(nào\s*\?*\s*)$',                    # "Nào?"
        r'^(đâu\s*\?*\s*)$'                     # "Đâu?"
    ]
    
    text_normalized = re.sub(r'\s+', ' ', text.lower().strip())
    for pattern in question_only_patterns:
        if re.match(pattern, text_normalized):
            return True, "question_word_only"
    
    # Check for single word + punctuation patterns (very short)
    if len(words) == 1 and len(text) <= 4:
        return True, "single_word_too_short"
    
    return False, ""


def save_annotation_with_guardrails(
    db,
    sample,
    pred,
    *,
    min_conf_auto_approve=MIN_CONF_APPROVE_EXISTING,
    min_conf_allow_new_label=MIN_CONF_AUTO_ADD_NEW_LABEL,
    allow_auto_add_new_label=ALLOW_AUTO_ADD_NEW_LABEL,
    model_name=OPENAI_MODEL,
    verify_meta=None,
    persist=True,
):
    col_ann = db["intent_annotations"]
    now = datetime.now(timezone.utc)
    
    # STEP 1: Check if sentence is too ambiguous to annotate
    sentence = sample.get("sentence", "").strip()
    is_ambiguous, ambiguous_reason = is_sentence_too_ambiguous_to_annotate(sentence)
    
    if is_ambiguous:
        # Skip annotation for ambiguous sentences
        ann_doc = {
            "sample_id": sample.get("sample_id"),
            "product_id": sample.get("product_id"),
            "sentence": sentence,
            "category": sample.get("category", ""),
            "model": model_name,
            "intent": {
                "level_1": "",
                "level_2": "",
                "level_3": [],
            },
            "confidence": 0.0,
            "reasoning": f"Câu quá ngắn/mơ hồ để annotate: {ambiguous_reason}",
            "qa_status": "skipped_ambiguous",
            "ambiguous_reason": ambiguous_reason,
            "new_label_pending_review": False,
            "graph_upserted": False,
            "semantic_snap_similarity": None,
            "semantic_snap_intent_id": None,
            "taxonomy_semantically_snapped": False,
            "verifier_decision": None,
            "verifier_reason": None,
            "confidence_before_verify": None,
            "source": sample.get("source", "hasaki"),
            "updated_at": now,
        }
        if persist:
            col_ann.update_one({"sample_id": ann_doc["sample_id"]}, {"$set": ann_doc}, upsert=True)
        return ann_doc
    
    # STEP 2: Continue with normal annotation logic
    allowed = _collect_allowed_taxonomy(db)
    # Snap chuỗi L1/L2/L3 → tên trong graph (truoc_mua_hang vs trước ban, underscore vs space, v.v.)
    _snap_pred_to_canonical_taxonomy(pred, allowed)
    confidence = float(pred.get("confidence") or 0)
    valid_existing = _is_label_allowed(pred, allowed)

    semantic_snap_similarity = None
    semantic_snap_intent_id = None
    used_semantic_snap = False
    auto_create_rejected_reason = None

    # Nếu triple không khớp taxonomy: embed đường dẫn/slug prediction, snap vào intent gần nhất trong graph (đối xứng với embeddings đã có)
    if not valid_existing:
        parts = []
        for p in (pred.get("L1"), pred.get("L2"), pred.get("L3")):
            if p is not None and str(p).strip():
                parts.append(str(p).strip())
        slug_blob = ".".join(parts)
        l3_piece = (pred.get("L3") or "").strip()
        reasoning_snip = ((pred.get("reasoning") or "").strip())[:280]
        pred_text_for_snap = slug_blob or l3_piece or reasoning_snip
        sentence_ctx = (sample.get("sentence") or "").strip()
        cat_ctx = (sample.get("category") or "").strip()
        extra_ctx = (" ".join(x for x in (sentence_ctx, cat_ctx) if x)).strip()
        # Triple rỗng hoặc slug không đủ: dùng câu hỏi thật để embed nearest intent (khớp case pending L1/L2/L3 trống)
        if not pred_text_for_snap and extra_ctx:
            pred_text_for_snap = extra_ctx[:512]
        elif extra_ctx and extra_ctx[:80] not in pred_text_for_snap:
            pred_text_for_snap = f"{pred_text_for_snap}\n{extra_ctx}".strip()[:768]

        nearest, nearest_sim = find_nearest_existing_intent(db, pred_text_for_snap)
        semantic_snap_similarity = float(nearest_sim)

        if nearest is not None:
            semantic_snap_intent_id = nearest.get("_id")
            pred["L1"] = nearest.get("l1") or ""
            pred["L2"] = nearest.get("l2") or ""
            pred["L3"] = nearest.get("l3") or ""
            _snap_pred_to_canonical_taxonomy(pred, allowed)
            valid_existing = _is_label_allowed(pred, allowed)
            used_semantic_snap = valid_existing

    qa_status = "rejected"
    new_label_pending_review = False
    graph_upserted = False

    if valid_existing:
        # Snap semantic: không phụ thuộc ngưỡng confidence gpt-4o-mini để thoát pending
        if used_semantic_snap:
            qa_status = "approved_snapped"
            new_label_pending_review = False
        else:
            qa_status = "approved" if confidence >= min_conf_auto_approve else "needs_review"
    else:
        new_label_pending_review = True
        qa_status = "pending_new_label_review"
        auto_create_rejected_reason = None
        if allow_auto_add_new_label and confidence >= min_conf_allow_new_label:
            valid_for_create, reject_reason, norm = _validate_pred_for_auto_create(pred)
            if valid_for_create:
                # Sử dụng L1/L2/L3 đã chuẩn hóa (truoc_mua_hang / sau_mua_hang + slug sạch)
                if persist:
                    upsert_new_label_to_graph(
                        db,
                        domain=sample.get("domain") or infer_domain(sample) or "Unknown",
                        l1=norm["L1"],
                        l2=norm["L2"],
                        l3=norm["L3"],
                        intent_id=sample.get("intent_id") or f"AUTO-{sample.get('sample_id', 'UNKNOWN')}",
                        intent_name=norm["L3"] or "new_intent",
                        description=pred.get("reasoning", ""),
                        confidence=str(confidence),
                        product_category=sample.get("category", ""),
                        detection_signals=sample.get("sentence", ""),
                        example=sample.get("sentence", ""),
                        category_logic="auto_added_from_labeling_notebook",
                    )
                    graph_upserted = True
                # Cập nhật pred để annotation lưu đúng slug đã normalize
                pred["L1"], pred["L2"], pred["L3"] = norm["L1"], norm["L2"], norm["L3"]
                qa_status = "approved_auto_new_label"
                new_label_pending_review = False
            else:
                # Conf cao nhưng L1/L2/L3 sai rule → giữ pending để human review,
                # KHÔNG upsert vào graph (tránh sinh node l1:gia_so_sanh, l1:trước bán, ...)
                auto_create_rejected_reason = reject_reason
                qa_status = "pending_new_label_invalid_slug"

    ann_doc = {
        "sample_id": sample.get("sample_id"),
        "product_id": sample.get("product_id"),
        "sentence": sample.get("sentence", ""),
        "category": sample.get("category", ""),
        "model": model_name,
        "intent": {
            "level_1": pred.get("L1", "").strip(),
            "level_2": pred.get("L2", "").strip(),
            "level_3": [pred.get("L3", "").strip()] if pred.get("L3") else [],
        },
        "confidence": confidence,
        "reasoning": pred.get("reasoning", ""),
        "qa_status": qa_status,
        "new_label_pending_review": new_label_pending_review,
        "graph_upserted": graph_upserted,
        "auto_create_rejected_reason": auto_create_rejected_reason,
        "semantic_snap_similarity": semantic_snap_similarity,
        "semantic_snap_intent_id": semantic_snap_intent_id,
        "taxonomy_semantically_snapped": used_semantic_snap,
        "verifier_decision": (verify_meta or {}).get("decision"),
        "verifier_reason": (verify_meta or {}).get("reason"),
        "confidence_before_verify": (verify_meta or {}).get("confidence_before_verify"),
        "source": sample.get("source", "hasaki"),
        "updated_at": now,
    }
    if persist:
        col_ann.update_one({"sample_id": ann_doc["sample_id"]}, {"$set": ann_doc}, upsert=True)
    return ann_doc


def _intent_slug_blob(d):
    parts = [d.get("l1"), d.get("l2"), d.get("l3"), d.get("name")]
    return " ".join(str(p or "") for p in parts).lower()


_ELECTRONICS_SLUG_MARKERS = (
    "dien_thoai",
    "laptop",
    "may_anh",
    "smartwatch",
    "tai_nghe",
    "iphone",
    "macbook",
    "gpu",
    "chuot",
)


def _candidate_looks_electronics_intent(d):
    s = _intent_slug_blob(d)
    return any(m in s for m in _ELECTRONICS_SLUG_MARKERS)


def _sentence_mentions_electronics(text):
    t = (text or "").lower()
    hints = (
        "dien thoai",
        "điện thoại",
        "laptop",
        "iphone",
        "macbook",
        "may anh",
        "máy ảnh",
        "tai nghe",
        "chuột",
        "ban phim",
        "bàn phím",
        "smartwatch",
        "card man hinh",
    )
    return any(h in t for h in hints)


def _category_match_bonus(sample, d):
    if not sample:
        return 0
    cat = (sample.get("category") or "").lower()
    pc = (d.get("product_category") or "").lower()
    if not cat or not pc:
        return 0
    cat_tokens = [w for w in re.findall(r"\w+", cat, flags=re.UNICODE) if len(w) >= 3]
    hit = sum(1 for w in cat_tokens if w in pc)
    # CHU Y: category chi dung lam tie-breaker, KHONG duoc lan at token match cua cau hoi.
    return min(1, hit)


def get_candidate_l3_from_mongodb(db, text, category=None, top_k=10, sample=None):
    tokens = [t.lower() for t in re.findall(r"\w+", text or "", flags=re.UNICODE) if len(t) >= 2]
    if not tokens:
        return []

    or_conditions = []
    for tok in tokens:
        rgx = {"$regex": re.escape(tok), "$options": "i"}
        or_conditions.extend(
            [
                {"name": rgx},
                {"description": rgx},
                {"detection_signals": rgx},
                {"example": rgx},
                {"l3": rgx},
            ]
        )

    query = {"level": "intent", "$or": or_conditions}
    docs = list(
        db[COL_NODES].find(
            query,
            {
                "_id": 1,
                "name": 1,
                "l1": 1,
                "l2": 1,
                "l3": 1,
                "description": 1,
                "detection_signals": 1,
                "example": 1,
                "product_category": 1,
            },
        )
    )

    beauty_ctx = bool(sample) and infer_domain(sample) == "My pham"

    scored = []
    token_set = set(tokens)
    for d in docs:
        if beauty_ctx and _candidate_looks_electronics_intent(d) and not _sentence_mentions_electronics(text):
            continue

        blob = " ".join(
            [
                str(d.get("name", "")),
                str(d.get("description", "")),
                str(d.get("detection_signals", "")),
                str(d.get("example", "")),
                str(d.get("l3", "")),
                str(d.get("product_category", "")),
            ]
        ).lower()
        hit = sum(1 for t in token_set if t in blob)
        if hit <= 0:
            continue
        cat_bonus = _category_match_bonus(sample, d) if sample else 0
        # Ngu nghia (token hits trong sentence/desc/signals) van la chu yeu;
        # category chi cong them <= 0.2 de tie-break giua 2 candidate ngang diem.
        scored.append((hit + 0.2 * cat_bonus, d))

    scored.sort(key=lambda x: x[0], reverse=True)
    pool = [d for _, d in scored[: max(top_k * 4, top_k + 8)]]

    seen = set()
    uniq = []
    for d in pool:
        key = (d.get("l1"), d.get("l2"), d.get("l3"))
        if key in seen:
            continue
        seen.add(key)
        uniq.append(d)
        if len(uniq) >= top_k:
            break
    return uniq


def build_retrieval_prompt(sample, candidates):
    vertical = infer_domain(sample)
    lines = []
    for i, c in enumerate(candidates, 1):
        pc = c.get("product_category") or ""
        lines.append(
            f"{i}. L1={c.get('l1')} | L2={c.get('l2')} | L3={c.get('l3')} | product_category={pc!r} | intent_id={c.get('_id')}"
        )
    taxonomy_text = (
        "\n".join(lines)
        if lines
        else "KHÔNG CÓ ỨNG VIÊN nào từ MongoDB. Bạn vẫn PHẢI đề xuất bộ (L1, L2, L3) hợp lệ theo quy tắc bên dưới và đặt is_new_label=true."
    )

    parts = [
        "Bạn là chuyên gia gán nhãn intent cho QA thương mại điện tử (mỹ phẩm / làm đẹp và điện tử).",
        "Nhiệm vụ: chọn intent dựa trên NGỮ NGHĨA câu hỏi của khách, KHÔNG dựa vào category/tên sản phẩm.",
        f"Ngành ước lượng (chỉ dùng để loại trừ slug ngành sai, KHÔNG dùng để chọn intent): **{vertical}**.",
        "",
        "QUY TẮC TAXONOMY (BẮT BUỘC):",
        "- L1 ∈ {'truoc_mua_hang', 'sau_mua_hang'} — viết đúng underscore + lowercase ASCII, KHÔNG khoảng trắng, KHÔNG dấu tiếng Việt.",
        "  + truoc_mua_hang: khách hỏi TRƯỚC khi mua (tư vấn, thành phần, công dụng, giá, khuyến mãi, tồn kho, bảo quản, an toàn da, phù hợp đối tượng).",
        "  + sau_mua_hang: khách đã/đang sở hữu (giao hàng, đổi trả, hoàn tiền, sản phẩm lỗi/hư khi nhận, khiếu nại dịch vụ, có mã đơn).",
        "- L2, L3 là slug ASCII lowercase + underscore (vd: di_ung_an_toan, chong_anh_sang_xanh). KHÔNG dấu, KHÔNG khoảng trắng.",
        "- L3 phải KHÁC L2 và mô tả ý định cụ thể hơn L2.",
        "",
        "CÁCH SUY LUẬN (BẮT BUỘC THEO THỨ TỰ NÀY):",
        "  1. Đọc kỹ NGỮ NGHĨA câu hỏi của khách (sentence) — xác định ý định chính: khách đang HỎI GÌ, MUỐN BIẾT GÌ.",
        "  2. Tìm L3 nào MÔ TẢ ĐÚNG ý định đó trong danh sách ứng viên (so khớp description / detection_signals / examples — KHÔNG so với product_category).",
        "  3. Suy ra L2 từ L3 đã chọn (sao chép từ ứng viên).",
        "  4. Quyết định L1 dựa vào tone/từ khóa câu (mã đơn / 'đã nhận' → sau_mua_hang; 'có … không' / 'phù hợp' → truoc_mua_hang).",
        "  5. CHỈ SAU CÙNG dùng category/product_name để LOẠI TRỪ slug ngành rõ ràng sai (vd. dien_thoai_* / laptop_* khi sản phẩm là mỹ phẩm).",
        "",
        "QUY TẮC CHỌN NHÃN:",
        "- KHÔNG được chọn intent chỉ vì product_category trùng với category của mẫu — phải có ngữ nghĩa khớp với câu hỏi.",
        "- KHÔNG suy diễn intent từ tên sản phẩm khi câu hỏi không nhắc tới (vd. câu chỉ hỏi 'giá bao nhiêu' thì chọn intent về giá, KHÔNG chọn intent đặc thù của sản phẩm).",
        "- Nếu CÓ dòng ứng viên phù hợp ngữ nghĩa: copy CHÍNH XÁC L1/L2/L3 từ dòng đó, đặt is_new_label=false.",
        "- Nếu KHÔNG có dòng nào khớp ngữ nghĩa câu hỏi: đặt is_new_label=true VÀ tự đề xuất bộ (L1, L2, L3) hợp lệ theo quy tắc taxonomy. KHÔNG ĐƯỢC để L1/L2/L3 rỗng — kể cả khi confidence thấp.",
        "- Tránh slug điện tử (dien_thoai_*, laptop_*, may_anh_*, tai_nghe_*…) khi câu KHÔNG nhắc rõ thiết bị điện tử.",
        "- Confidence ∈ [0,1]: ≥0.90 rất chắc (ngữ nghĩa khớp 1-1), 0.70–0.89 hợp lý, <0.70 không chắc. KHÔNG trả 0.0 nếu đã đề xuất được L1/L2/L3.",
        "",
        "ỨNG VIÊN (MongoDB):",
        taxonomy_text,
        "",
        "NGỮ CẢNH ĐẦU VÀO (sentence là CHÍNH; category/product_name/brand chỉ là context phụ — chỉ dùng để loại trừ ngành sai):",
        f"- Câu (sentence) [CHÍNH]: {sample.get('sentence', '')}",
        f"- Danh mục (category) [phụ]: {sample.get('category', '')}",
        f"- Tên sản phẩm (product_name) [phụ]: {sample.get('product_name', '')}",
        f"- Thương hiệu (brand) [phụ]: {sample.get('brand', '')}",
        "",
        "Trả về DUY NHẤT một đối tượng JSON hợp lệ (KHÔNG markdown, KHÔNG text ngoài JSON).",
        "Thứ tự khóa — bắt buộc: viết reasoning trước (1–3 câu tiếng Việt), sau đó confidence và nhãn:",
        "{",
        '  "reasoning": "...",',
        '  "confidence": 0.0,',
        '  "L1": "...",',
        '  "L2": "...",',
        '  "L3": "...",',
        '  "is_new_label": false',
        "}",
    ]
    return "\n".join(parts)

_HASAKI_BEAUTY_CATEGORIES_CACHE = None  # (frozenset, mtime_ns) sau khi load thanh cong


def get_hasaki_beauty_categories():
    """Doc unique `category` tu HASAKI_JSON; cache theo mtime file (doi file -> doc lai)."""
    global _HASAKI_BEAUTY_CATEGORIES_CACHE
    path = HASAKI_JSON
    mtime = path.stat().st_mtime_ns if path.exists() else -1
    if isinstance(_HASAKI_BEAUTY_CATEGORIES_CACHE, tuple) and len(_HASAKI_BEAUTY_CATEGORIES_CACHE) == 2:
        cached, mt = _HASAKI_BEAUTY_CATEGORIES_CACHE
        if mt == mtime:
            return cached
    cats = set()
    try:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                for row in data:
                    if isinstance(row, dict):
                        c = (row.get("category") or "").strip()
                        if c:
                            cats.add(c)
    except Exception:
        pass
    if not cats:
        cats = {
            "Sức khỏe làm đẹp",
            "Trang điểm",
            "Mỹ phẩm high end",
        }
    fs = frozenset(cats)
    _HASAKI_BEAUTY_CATEGORIES_CACHE = (fs, mtime)
    return fs


def clear_hasaki_beauty_category_cache():
    """Xoa cache catalog category (sau khi doi HASAKI_JSON hoac can ep doc lai)."""
    global _HASAKI_BEAUTY_CATEGORIES_CACHE
    _HASAKI_BEAUTY_CATEGORIES_CACHE = None


def infer_domain(sample):
    cat_raw = (sample.get("category") or "").strip()
    cat = cat_raw.lower()
    name = (sample.get("product_name") or "").lower()
    blob = f"{cat} {name}"

    if cat_raw and cat_raw in get_hasaki_beauty_categories():
        return "My pham"

    electronics_kw = (
        "laptop phone dien thoai tai nghe chuot ban phim camera smartwatch "
        "gaming console ram gpu cpu may anh dong ho thong minh "
        "tablet iphone tivi tv"
    )
    if any(k in blob for k in electronics_kw.split()):
        return "Dien tu"

    cat_hints = (
        "may tinh",
        "dien tu",
        "phu kien dien",
        "dien thoai",
        "dong ho thong minh",
    )
    if any(h in cat for h in cat_hints):
        return "Dien tu"

    return "My pham"


print("Helpers OK")


Helpers OK


## 5. LLM client (OpenAI gpt-4o-mini)

Cell bên dưới khởi tạo `openai_client` cho `OPENAI_MODEL` (mặc định `gpt-4o-mini`).

- Cần `OPENAI_API_KEY` (§2.5) trước khi chạy batch.

`predict_intent()` gọi `_call_llm()` → OpenAI Chat Completions.


In [ ]:
# LLM client — OpenAI gpt-4o-mini (chạy cell này trước §6 / §8)
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ OpenAI API model: {OPENAI_MODEL}")


## 6. Sinh nhãn + parse JSON

- **Annotator:** `predict_intent()` — gpt-4o-mini chọn (L1, L2, L3) + confidence.
- **Guardrail:** `save_annotation_with_guardrails()` kiểm taxonomy, semantic snap, auto-create label và gán `qa_status`.
- **Human review:** pipeline dừng ở `qa_status`; `needs_review` và `pending_new_label_review` được đưa cho người rà soát.


In [ ]:
import json as json_lib
import re
import time


SYSTEM_PROMPT = (
    "Bạn chỉ trả về một đoạn JSON hợp lệ, không markdown, không text ngoài JSON. "
    "Tuân thủ thứ tự khóa trong yêu cầu: reasoning trước, sau đó confidence, L1, L2, L3, is_new_label."
)


def _call_llm(messages, max_new_tokens=None, model=None):
    """Gọi OpenAI Chat Completions, trả raw text (JSON string).

    model=None -> dùng OPENAI_MODEL (annotator). Truyền model khác (vd VERIFIER_MODEL)
    để gọi verifier mà không phá Phase 1.
    """
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS
    if model is None:
        model = OPENAI_MODEL
    if openai_client is None:
        raise RuntimeError(
            "openai_client chưa khởi tạo — chạy lại cell §5 (LLM client setup)."
        )
    last_err = None
    for attempt in range(OPENAI_MAX_RETRIES):
        try:
            resp = openai_client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=OPENAI_TEMPERATURE,
                max_tokens=max_new_tokens,
                response_format={"type": "json_object"},
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"OpenAI call thất bại sau {OPENAI_MAX_RETRIES} lần: {last_err}")



def extract_json_object(text):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    if "```" in text:
        for chunk in text.split("```"):
            c = chunk.strip()
            if c.lower().startswith("json"):
                c = c[4:].strip()
            if c.startswith("{"):
                text = c
                break
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Khong tim thay JSON")
    return json_lib.loads(text[start : end + 1])


def predict_intent(sample, top_k=None, use_semantic=True):
    """Enhanced predict_intent with graph-aware semantic retrieval and ambiguity filtering
    
    Args:
        sample: sample data dict
        top_k: max candidates (fallback to TOP_K_CANDIDATES)
        use_semantic: whether to use union retrieval (True) or regex only (False)
    
    Returns:
        tuple: (prediction, raw_output, candidates)
    """
    if top_k is None:
        top_k = TOP_K_CANDIDATES
    
    # PRE-CHECK: Skip prediction if sentence is too ambiguous
    sentence = sample.get("sentence", "").strip()
    is_ambiguous, ambiguous_reason = is_sentence_too_ambiguous_to_annotate(sentence)
    
    if is_ambiguous:
        # Return dummy prediction for ambiguous sentences
        pred = {
            "reasoning": f"Câu quá ngắn/mơ hồ để dự đoán: {ambiguous_reason}",
            "confidence": 0.0,
            "L1": "",
            "L2": "",
            "L3": "",
            "is_new_label": False,
            "_ambiguous": True,
            "_ambiguous_reason": ambiguous_reason
        }
        return pred, f"SKIPPED_AMBIGUOUS: {ambiguous_reason}", []
    
    # INTEGRATION POINT: Use union retrieval instead of regex-only
    if use_semantic:
        cands = union_retrieval(
            db, 
            sample.get("sentence", ""), 
            category=sample.get("category"),
            top_k_regex=max(6, top_k//2),
            top_k_semantic=max(4, top_k//3),
            sample=sample
        )
    else:
        # Fallback to original regex retrieval
        cands = get_candidate_l3_from_mongodb(
            db, sample.get("sentence", ""), category=sample.get("category"), top_k=top_k, sample=sample
        )
    
    user_prompt = build_retrieval_prompt(sample, cands)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    raw = _call_llm(messages, max_new_tokens=MAX_NEW_TOKENS)
    pred = extract_json_object(raw)
    return pred, raw, cands


def enrich_sample(s):
    out = dict(s)
    out["domain"] = infer_domain(s)
    out["intent_id"] = f"AUTO-{s.get('sample_id', 'x')}"
    return out


# === §6.5. Verifier (cross-verification, GPT-4o) ===
# Doc lai nhan annotator (gpt-4o-mini) bang VERIFIER_MODEL (mac dinh gpt-4o) -> RETAIN | REVISE.
# Verifier KHONG enforce taxonomy; de guardrail (§4) xu ly.

VERIFIER_SYSTEM_PROMPT = (
    "Bạn là model kiểm định nhãn intent cho QA thương mại điện tử (mỹ phẩm / điện tử). "
    "Bạn chỉ trả về một đoạn JSON hợp lệ, không markdown, không text ngoài JSON."
)


def build_verification_prompt(sample, pred, candidates):
    lines = []
    for i, c in enumerate(candidates or [], 1):
        pc = c.get("product_category") or ""
        lines.append(
            f"{i}. L1={c.get('l1')} | L2={c.get('l2')} | L3={c.get('l3')} | product_category={pc!r}"
        )
    taxonomy_text = "\n".join(lines) if lines else "KHÔNG CÓ ứng viên từ MongoDB."

    parts = [
        "Nhiệm vụ: KIỂM ĐỊNH bộ nhãn (L1, L2, L3) mà annotator đã gán cho câu hỏi của khách.",
        "Đánh giá dựa trên NGỮ NGHĨA câu hỏi, KHÔNG dựa vào category/tên sản phẩm.",
        "",
        "QUY TẮC TAXONOMY (BẮT BUỘC):",
        "- L1 ∈ {'truoc_mua_hang', 'sau_mua_hang'} — lowercase ASCII + underscore, không dấu, không khoảng trắng.",
        "  + truoc_mua_hang: hỏi TRƯỚC khi mua (tư vấn, thành phần, công dụng, giá, khuyến mãi, tồn kho, an toàn da, phù hợp đối tượng).",
        "  + sau_mua_hang: đã/đang sở hữu (giao hàng, đổi trả, hoàn tiền, sản phẩm lỗi khi nhận, khiếu nại, có mã đơn).",
        "- L2, L3 là slug ASCII lowercase + underscore. L3 phải KHÁC L2 và cụ thể hơn.",
        "",
        "CÁCH QUYẾT ĐỊNH:",
        "- Nếu nhãn annotator khớp đúng ngữ nghĩa câu hỏi → decision='RETAIN', giữ nguyên final_L1/L2/L3.",
        "- Nếu sai (sai L1 truoc/sau, sai slug, hiểu nhầm category) → decision='REVISE' và đề xuất bộ (L1, L2, L3) đúng hơn.",
        "- KHÔNG để final_L1/L2/L3 rỗng. confidence_adjusted ∈ [0,1] phản ánh độ tin sau khi kiểm định.",
        "",
        "ỨNG VIÊN (MongoDB):",
        taxonomy_text,
        "",
        "CÂU HỎI KHÁCH (sentence là CHÍNH; context phụ chỉ để loại trừ ngành sai):",
        f"- Câu (sentence): {sample.get('sentence', '')}",
        f"- Danh mục (category) [phụ]: {sample.get('category', '')}",
        f"- Tên sản phẩm (product_name) [phụ]: {sample.get('product_name', '')}",
        "",
        "NHÃN ANNOTATOR ĐỀ XUẤT (cần kiểm định):",
        f"- L1: {pred.get('L1', '')}",
        f"- L2: {pred.get('L2', '')}",
        f"- L3: {pred.get('L3', '')}",
        f"- confidence: {pred.get('confidence', '')}",
        f"- reasoning: {pred.get('reasoning', '')}",
        "",
        "Trả về DUY NHẤT một JSON hợp lệ theo đúng thứ tự khóa:",
        "{",
        '  "decision": "RETAIN",',
        '  "final_L1": "...",',
        '  "final_L2": "...",',
        '  "final_L3": "...",',
        '  "confidence_adjusted": 0.0,',
        '  "reason": "giải thích ngắn gọn tiếng Việt"',
        "}",
    ]
    return "\n".join(parts)


def verify_intent(sample, pred, candidates):
    """Goi verifier (VERIFIER_MODEL) doc lai pred. Tra dict chuan hoa.

    Output keys: decision (RETAIN|REVISE), final_L1/L2/L3, confidence_adjusted, reason.
    Neu verifier loi/parse fail -> fallback RETAIN giu nguyen pred (an toan).
    """
    user_prompt = build_verification_prompt(sample, pred, candidates)
    messages = [
        {"role": "system", "content": VERIFIER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        raw = _call_llm(messages, max_new_tokens=VERIFY_MAX_TOKENS, model=VERIFIER_MODEL)
        v = extract_json_object(raw)
    except Exception as e:
        return {
            "decision": "RETAIN",
            "final_L1": pred.get("L1", ""),
            "final_L2": pred.get("L2", ""),
            "final_L3": pred.get("L3", ""),
            "confidence_adjusted": float(pred.get("confidence") or 0),
            "reason": f"verifier_error: {e}",
        }

    decision = str(v.get("decision") or "RETAIN").strip().upper()
    if decision not in ("RETAIN", "REVISE"):
        decision = "RETAIN"
    try:
        conf_adj = float(v.get("confidence_adjusted"))
    except (TypeError, ValueError):
        conf_adj = float(pred.get("confidence") or 0)
    conf_adj = max(0.0, min(1.0, conf_adj))

    if decision == "REVISE":
        f1 = (v.get("final_L1") or "").strip()
        f2 = (v.get("final_L2") or "").strip()
        f3 = (v.get("final_L3") or "").strip()
        # Verifier thieu nhan -> coi nhu RETAIN de tranh lam rong pred.
        if not (f1 and f2 and f3):
            decision = "RETAIN"
            f1, f2, f3 = pred.get("L1", ""), pred.get("L2", ""), pred.get("L3", "")
    else:
        f1, f2, f3 = pred.get("L1", ""), pred.get("L2", ""), pred.get("L3", "")

    return {
        "decision": decision,
        "final_L1": f1,
        "final_L2": f2,
        "final_L3": f3,
        "confidence_adjusted": conf_adj,
        "reason": (v.get("reason") or "").strip(),
    }


def apply_verifier(sample, pred, candidates, *, force=False):
    """Goi verifier neu can va merge ket qua vao pred (in-place).

    Tra ve verify_meta (dict) hoac None neu khong verify.
    Chi verify khi: ENABLE_VERIFIER, khong _ambiguous, va (force hoac VERIFY_ALL_SAMPLES hoac confidence < VERIFY_CONF_THRESHOLD).
    """
    if not ENABLE_VERIFIER:
        return None
    if pred.get("_ambiguous"):
        return None
    conf0 = float(pred.get("confidence") or 0)
    if not force and not VERIFY_ALL_SAMPLES and conf0 >= VERIFY_CONF_THRESHOLD:
        return None

    v = verify_intent(sample, pred, candidates)
    v["confidence_before_verify"] = conf0
    if v["decision"] == "REVISE":
        pred["L1"], pred["L2"], pred["L3"] = v["final_L1"], v["final_L2"], v["final_L3"]
        pred["confidence"] = v["confidence_adjusted"]
        if v.get("reason"):
            pred["reasoning"] = (pred.get("reasoning", "") + " | VERIFIER: " + v["reason"]).strip()
    else:  # RETAIN
        pred["confidence"] = max(conf0, v["confidence_adjusted"])
    return v


# Pipeline dừng ở nhãn + qa_status. Các mẫu `needs_review` và
# `pending_new_label_review` được human review ngoài notebook.


## 7. Thử một mẫu


In [ ]:
# Embed intent nodes (bắt buộc trước batch). FORCE_REEMBED=1 nếu vừa rebuild graph.
FORCE_REEMBED = bool(int(os.environ.get("FORCE_REEMBED", "0")))
embed_count = embed_and_store_intent_nodes(db, force=FORCE_REEMBED)
print(f"Embedded nodes (new={embed_count}, force={FORCE_REEMBED})")

# Thử một mẫu end-to-end: retrieve → annotate → guardrail
sample_demo = enrich_sample({
    "sample_id": "demo_001",
    "product_name": "Kem dưỡng da",
    "brand": "CeraVe",
    "category": "Chăm sóc da",
    "sentence": "Sản phẩm này có bị hỏng do nắng nóng không?",
    "source": "demo",
})

pred, raw, cands = predict_intent(sample_demo, use_semantic=True)
print(f"Candidates: {len(cands)}")
for i, c in enumerate(cands[:5], 1):
    print(f"  {i}. {c.get('l1')}.{c.get('l2')}.{c.get('l3')}")
print("Prediction:", pred.get("L1"), pred.get("L2"), pred.get("L3"), "| conf:", pred.get("confidence"))

ann = save_annotation_with_guardrails(db, sample_demo, pred)
print("qa_status:", ann.get("qa_status"))


## 7.5. Ghi chú verifier + human review

Pipeline 2 bước (§6.5): **Annotate (gpt-4o-mini)** → **Verify (gpt-4o)** → guardrail. Verifier chạy khi `VERIFY_ALL_SAMPLES=1` (mọi mẫu, khớp diagram) hoặc mặc định chỉ khi `confidence < VERIFY_CONF_THRESHOLD`. Kết quả: `RETAIN` hoặc `REVISE`; metadata ở `verifier_decision` / `verifier_reason` / `confidence_before_verify`. Tắt bằng `ENABLE_VERIFIER=0`.

Sau khi export, human review tập trung vào các mẫu có `qa_status` là `needs_review` hoặc `pending_new_label_review`.


In [ ]:
# Demo verifier trên một câu confidence thấp (chạy sau §6 + §5 LLM client).
# Verifier chỉ kích hoạt khi confidence < VERIFY_CONF_THRESHOLD.
_demo_sample = enrich_sample({
    "sample_id": "demo_verify_1",
    "sentence": "giá ở cửa hàng có giống trên app không ạ",
    "category": "Sức khỏe làm đẹp",
})
try:
    _p, _raw, _c = predict_intent(_demo_sample)
    print("Annotator:", _p.get("L1"), _p.get("L2"), _p.get("L3"), "| conf:", _p.get("confidence"))
    _vm = apply_verifier(_demo_sample, _p, _c)
    if _vm is None:
        print("→ Bỏ qua verifier (ambiguous hoặc conf ≥ ngưỡng).")
    else:
        print(f"Verifier [{_vm['decision']}]:", _p.get("L1"), _p.get("L2"), _p.get("L3"),
              "| conf:", _p.get("confidence"), "| reason:", _vm.get("reason"))
except Exception as _e:
    print("Demo verifier cần §3 (MongoDB) + §5 (LLM client) chạy trước:", _e)

print("Human review statuses:", sorted(HUMAN_REVIEW_QA_STATUSES))


## 8. Batch Hasaki với Graph-Aware Semantic Retrieval

- **Chạy full file:** cell `hasaki-full-batch-run` — `json.load` một lần, xử lý theo **`HASAKI_BATCH_SIZE`**, checkpoint `data/outputs/hasaki_ckpt_*.json`. Sau khi xong: **giải phóng** list toàn file (`del hasaki_all`) để nhường RAM.
- **Resume:** trong cell **§2 Cấu hình** đặt `HASAKI_RUN_OFFSET` = index bắt đầu (sau đó chạy lại cell full).
- **Export:** cell tiếp theo ghi `hasaki_labelled_full_n*_offset*.json`. Có thể bật `HASAKI_FREE_MEM_AFTER_EXPORT` (§2) để **xóa `BATCH_OUTPUT` khỏi RAM** sau khi ghi file (kết quả vẫn trên đĩa).
- **Cache category:** `get_hasaki_beauty_categories()` cache theo **mtime** file prelabel (bộ nhớ nhỏ); gọi `clear_hasaki_beauty_category_cache()` khi đổi file prelabel và cần ép đọc lại.
- **Semantic Retrieval:** Mặc định sử dụng union retrieval (regex + semantic) để giảm `pending_new_label_review`.


In [ ]:
def run_batch(samples):
    stats = {}
    results = []
    for s in samples:
        ex = enrich_sample(s)
        try:
            # Annotator + RAG candidates; human review dựa trên qa_status sau guardrail.
            pred, raw, cands = predict_intent(ex)
            # §6.5 Verifier: chỉ chạy khi confidence < VERIFY_CONF_THRESHOLD (merge in-place).
            verify_meta = apply_verifier(ex, pred, cands)
            ann = save_annotation_with_guardrails(db, ex, pred, verify_meta=verify_meta)
            st = ann["qa_status"]
            stats[st] = stats.get(st, 0) + 1
            if verify_meta is not None:
                vkey = f"verify_{verify_meta['decision'].lower()}"
                stats[vkey] = stats.get(vkey, 0) + 1
            results.append(
                {
                    "sample_id": ex.get("sample_id"),
                    "sentence": ex.get("sentence"),
                    "category": ex.get("category"),
                    "prediction": pred,
                    "annotation": ann,
                    "verification": verify_meta,
                    "candidate_count": len(cands),
                }
            )
            vtag = f" [{verify_meta['decision']}]" if verify_meta else ""
            print(st, ex.get("sample_id"), vtag)
        except Exception as e:
            stats["error"] = stats.get("error", 0) + 1
            results.append(
                {
                    "sample_id": ex.get("sample_id"),
                    "sentence": ex.get("sentence"),
                    "category": ex.get("category"),
                    "error": str(e),
                }
            )
            print("ERROR", ex.get("sample_id"), e)
    return {"stats": stats, "results": results}


# --- Chay HET hasaki_prelabel ---
# CANH: HASAKI_JSON, HASAKI_BATCH_SIZE, HASAKI_RUN_OFFSET o cell "2. Cau hinh"
# OUTPUT_DIR đã được set ở cell §2 (mặc định REPO_ROOT/data/outputs, trên Colab nằm trong Drive).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("📂 Checkpoint sẽ ghi vào:", OUTPUT_DIR)

if not HASAKI_JSON.exists():
    print("Khong tim thay HASAKI_JSON:", HASAKI_JSON)
elif HASAKI_BATCH_SIZE <= 0:
    raise ValueError("HASAKI_BATCH_SIZE phai > 0")
else:
    with open(HASAKI_JSON, "r", encoding="utf-8") as f:
        hasaki_all = json_lib.load(f)
    n = len(hasaki_all)
    off = int(HASAKI_RUN_OFFSET)
    bs = int(HASAKI_BATCH_SIZE)
    if off < 0 or off >= n:
        raise ValueError(f"HASAKI_RUN_OFFSET={off} khong hop le (tong mau {n})")
    print("Tong mau:", n, "| batch_size:", bs, "| offset:", off)

    agg_stats = {}
    all_results = []
    for start in range(off, n, bs):
        chunk = hasaki_all[start : start + bs]
        print("=== chunk", start, "..", start + len(chunk) - 1, f"({len(chunk)} mau)")
        out = run_batch(chunk)
        for k, v in out["stats"].items():
            agg_stats[k] = agg_stats.get(k, 0) + v
        all_results.extend(out["results"])
        ckpt = OUTPUT_DIR / f"hasaki_ckpt_{start:05d}_{start + len(chunk):05d}.json"
        with open(ckpt, "w", encoding="utf-8") as cf:
            json_lib.dump(
                {
                    "chunk_start": start,
                    "chunk_end": start + len(chunk),
                    "stats": out["stats"],
                    "results": out["results"],
                },
                cf,
                ensure_ascii=False,
                indent=2,
                default=str,
            )
        print("checkpoint:", ckpt)
        del out
        del chunk

    BATCH_OUTPUT = {"stats": agg_stats, "results": all_results}
    HASAKI_LAST_RUN = {
        "total_in_file": n,
        "labelled": len(all_results),
        "offset": off,
        "batch_size": bs,
    }
    # Giai phong list toan file khoi RAM (ket qua da gom trong BATCH_OUTPUT + tung ckpt)
    del hasaki_all
    batch = None
    import gc

    gc.collect()
    print("Xong full file. Tong stats:", agg_stats)
    print("Can human review:", {k: agg_stats.get(k, 0) for k in HUMAN_REVIEW_QA_STATUSES})
    print("So mau da label:", len(all_results), "| HASAKI_LAST_RUN:", HASAKI_LAST_RUN)
    print("Da xoa hasaki_all / chunk khoi RAM (cache category Hasaki van giu — nhe).")



In [ ]:
# Xuat JSON sau khi chay full hasaki_prelabel (cell "hasaki-full-batch-run")
# OUTPUT_DIR đã được set ở cell §2 (mặc định REPO_ROOT/data/outputs, trên Colab nằm trong Drive).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH_OUTPUT" not in globals():
    print("Chua co BATCH_OUTPUT — hay chay cell full hasaki (hasaki-full-batch-run) truoc.")
elif not BATCH_OUTPUT.get("results"):
    print("BATCH_OUTPUT khong co results — kiem tra loi khi chay full file.")
else:
    nr = len(BATCH_OUTPUT["results"])
    off = int(HASAKI_RUN_OFFSET) if "HASAKI_RUN_OFFSET" in globals() else 0
    OUTPUT_FILE = OUTPUT_DIR / f"hasaki_labelled_full_n{nr}_offset{off}.json"

    payload = {
        "run_mode": "full_hasaki_prelabel",
        "hasaki_json": str(HASAKI_JSON),
        "hasaki_exists": HASAKI_JSON.exists(),
        "hasaki_batch_size": HASAKI_BATCH_SIZE,
        "hasaki_run_offset": off,
        "num_labelled": nr,
    }
    if "HASAKI_LAST_RUN" in globals():
        payload["last_run"] = HASAKI_LAST_RUN
    if "HASAKI_LAST_RUN" in globals() and isinstance(HASAKI_LAST_RUN, dict):
        payload["num_samples_in_source"] = HASAKI_LAST_RUN.get("total_in_file")
    elif "batch" in globals() and batch is not None:
        payload["num_samples_in_source"] = len(batch)

    payload["stats"] = BATCH_OUTPUT.get("stats", {})
    payload["human_review_statuses"] = sorted(HUMAN_REVIEW_QA_STATUSES)
    payload["human_review_counts"] = {
        k: BATCH_OUTPUT.get("stats", {}).get(k, 0) for k in HUMAN_REVIEW_QA_STATUSES
    }

    formatted_results = []
    for item in BATCH_OUTPUT.get("results", []):
        pred = item.get("prediction") or {}
        ann = item.get("annotation") or {}
        ann_intent = ann.get("intent") or {}

        level_1 = pred.get("L1") or ann_intent.get("level_1") or ""
        level_2 = pred.get("L2") or ann_intent.get("level_2") or ""
        level_3 = pred.get("L3")
        if not level_3:
            l3_list = ann_intent.get("level_3") or []
            level_3 = l3_list[0] if l3_list else ""

        formatted_results.append(
            {
                "sample_id": item.get("sample_id"),
                "sentence": item.get("sentence"),
                "category": item.get("category"),
                "intent_3_level": {
                    "level_1": level_1,
                    "level_2": level_2,
                    "level_3": level_3,
                },
                "confidence": ann.get("confidence", pred.get("confidence")),
                "qa_status": ann.get("qa_status"),
                "needs_human_review": ann.get("qa_status") in HUMAN_REVIEW_QA_STATUSES,
                "reasoning": pred.get("reasoning", ann.get("reasoning")),
                "verifier_decision": ann.get("verifier_decision"),
                "verifier_reason": ann.get("verifier_reason"),
                "confidence_before_verify": ann.get("confidence_before_verify"),
                "error": item.get("error"),
            }
        )

    payload["results"] = formatted_results

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json_lib.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Da tao file:", OUTPUT_FILE)
    print("So ket qua da luu:", len(payload.get("results", [])))

    if globals().get("HASAKI_FREE_MEM_AFTER_EXPORT", False):
        globals().pop("BATCH_OUTPUT", None)
        globals().pop("batch", None)
        import gc

        gc.collect()
        print("Da giai phong BATCH_OUTPUT / batch trong RAM (HASAKI_FREE_MEM_AFTER_EXPORT=True).")



## 8.5. Xuất hàng đợi human review

Pipeline RAG dừng ở `qa_status`. Cell này đọc file labelled đã có và xuất riêng các mẫu cần người rà soát:

- `needs_review`: nhãn hợp lệ trong taxonomy nhưng confidence thấp.
- `pending_new_label_review`: model đề xuất nhãn mới / chưa map được vào taxonomy hiện có.

Output: `*_human_review_queue.json`.


In [ ]:
# === Xuất hàng đợi human review từ file labelled đã có ===
REVIEW_INPUT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json"

if not REVIEW_INPUT_FILE.exists():
    raise FileNotFoundError(f"Khong thay file labelled: {REVIEW_INPUT_FILE}")

with open(REVIEW_INPUT_FILE, "r", encoding="utf-8") as f:
    _rin = json_lib.load(f)

_results_in = _rin.get("results", _rin) if isinstance(_rin, dict) else _rin
review_queue = [
    item for item in _results_in
    if (item.get("qa_status") or "") in HUMAN_REVIEW_QA_STATUSES
]

review_stats = {}
for item in review_queue:
    st = item.get("qa_status") or ""
    review_stats[st] = review_stats.get(st, 0) + 1

_rout = {
    "source_file": str(REVIEW_INPUT_FILE),
    "human_review_statuses": sorted(HUMAN_REVIEW_QA_STATUSES),
    "num_review_items": len(review_queue),
    "review_stats": review_stats,
    "results": review_queue,
}

REVIEW_OUTPUT_FILE = REVIEW_INPUT_FILE.with_name(REVIEW_INPUT_FILE.stem + "_human_review_queue.json")
with open(REVIEW_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json_lib.dump(_rout, f, ensure_ascii=False, indent=2, default=str)

print("=== Human review queue ===")
print("Review stats:", review_stats)
print("So mau can human review:", len(review_queue))
print("File output:", REVIEW_OUTPUT_FILE)


## 8.6. Reverify nhãn human review (dry-run, không ghi MongoDB)

Cell này **verify lại** chỉ các mẫu cần human review (`needs_review`, `pending_new_label_review` — cùng tập với §8.5), không chạy trên toàn bộ file (~2000 mẫu).

1. Đọc `REVERIFY_INPUT_FILE` (mặc định `hasaki_labelled_clean.json`).
2. Lọc theo `HUMAN_REVIEW_QA_STATUSES` (§2).
3. Với mỗi mẫu: `union_retrieval` → `apply_verifier(..., force=True)` → `save_annotation_with_guardrails(..., persist=False)`.
4. Ghi `*_human_review_reverified.json` — **không** ghi MongoDB.

**Prerequisite:** chạy §2 → §3 → §4 → §5 → §6 → §7 trước.

**So sánh trước/sau:** `qa_status_before_reverify`, `intent_3_level_before_reverify`, nhãn/`qa_status` sau reverify.

In [ ]:
# === Reverify human review queue (dry-run: persist=False, khong ghi MongoDB) ===
REVERIFY_INPUT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json"
REVERIFY_OUTPUT_FILE = REVERIFY_INPUT_FILE.with_name(
    REVERIFY_INPUT_FILE.stem + "_human_review_reverified.json"
)
# Chi reverify cac qa_status can human review (mac dinh: needs_review, pending_new_label_review)
REVERIFY_QA_STATUSES = HUMAN_REVIEW_QA_STATUSES

if not REVERIFY_INPUT_FILE.exists():
    raise FileNotFoundError(f"Khong thay file labelled: {REVERIFY_INPUT_FILE}")


def _intent_from_ann(ann_intent, fallback_intent=None):
    """Chuyển ann.intent (level_3 có thể là list) → intent_3_level export."""
    fb = fallback_intent or {}
    l3_raw = (ann_intent or {}).get("level_3")
    if isinstance(l3_raw, list):
        l3 = l3_raw[0] if l3_raw else ""
    elif isinstance(l3_raw, str):
        l3 = l3_raw
    else:
        fb3 = fb.get("level_3")
        l3 = fb3[0] if isinstance(fb3, list) and fb3 else (fb3 or "")
    return {
        "level_1": (ann_intent or {}).get("level_1") or fb.get("level_1", ""),
        "level_2": (ann_intent or {}).get("level_2") or fb.get("level_2", ""),
        "level_3": l3 or "",
    }


def _pred_from_labelled_item(item):
    intent = item.get("intent_3_level") or {}
    l3_val = intent.get("level_3")
    if isinstance(l3_val, list):
        l3_val = l3_val[0] if l3_val else ""
    return {
        "L1": intent.get("level_1", ""),
        "L2": intent.get("level_2", ""),
        "L3": l3_val or "",
        "confidence": float(item.get("confidence") or 0),
        "reasoning": item.get("reasoning", ""),
        "is_new_label": False,
    }


with open(REVERIFY_INPUT_FILE, "r", encoding="utf-8") as f:
    _rv_in = json_lib.load(f)

_rv_results_all = _rv_in.get("results", _rv_in) if isinstance(_rv_in, dict) else _rv_in
_rv_results_in = [
    item for item in _rv_results_all
    if (item.get("qa_status") or "") in REVERIFY_QA_STATUSES
]
reverify_stats = {}
reverified_results = []

print(f"Reverify human review: {len(_rv_results_in)}/{len(_rv_results_all)} mau tu {REVERIFY_INPUT_FILE.name}")
print(f"  qa_status filter: {sorted(REVERIFY_QA_STATUSES)}")
if not _rv_results_in:
    print("Khong co mau nao trong human review queue — kiem tra qa_status hoac file input.")
print(
    f"Verifier: {'ON' if ENABLE_VERIFIER else 'OFF'} | model={VERIFIER_MODEL} | force=True (bo qua nguong conf)"
)

for idx, item in enumerate(_rv_results_in):
    sample = enrich_sample({
        "sample_id": item.get("sample_id"),
        "sentence": item.get("sentence", ""),
        "category": item.get("category", ""),
    })
    intent_before = item.get("intent_3_level") or {}
    pred = _pred_from_labelled_item(item)
    qa_status_before = item.get("qa_status")
    verify_meta = None
    ann = {}
    status = qa_status_before
    reverify_error = None

    try:
        cands = union_retrieval(
            db,
            sample.get("sentence", ""),
            category=sample.get("category"),
            sample=sample,
        )
        verify_meta = apply_verifier(sample, pred, cands, force=True)
        ann = save_annotation_with_guardrails(
            db, sample, pred, verify_meta=verify_meta, persist=False
        )
        status = ann.get("qa_status", status)
    except Exception as e:
        reverify_error = str(e)
        status = "reverify_error"

    reverify_stats[status] = reverify_stats.get(status, 0) + 1
    if verify_meta is not None:
        vkey = f"verify_{str(verify_meta.get('decision', '')).lower()}"
        reverify_stats[vkey] = reverify_stats.get(vkey, 0) + 1
    elif verify_meta is None and ENABLE_VERIFIER and status != "reverify_error":
        reverify_stats["verify_skipped"] = reverify_stats.get("verify_skipped", 0) + 1

    intent_after = _intent_from_ann(ann.get("intent"), fallback_intent=intent_before)

    reverified_results.append({
        **item,
        "qa_status_before_reverify": qa_status_before,
        "intent_3_level_before_reverify": intent_before,
        "intent_3_level": intent_after,
        "confidence": ann.get("confidence", pred.get("confidence")),
        "qa_status": status,
        "reasoning": pred.get("reasoning", item.get("reasoning")),
        "verifier_decision": (verify_meta or {}).get("decision"),
        "verifier_reason": (verify_meta or {}).get("reason"),
        "confidence_before_verify": (verify_meta or {}).get("confidence_before_verify"),
        "reverify_error": reverify_error,
    })

    if (idx + 1) % 200 == 0:
        print(f"  Processed {idx + 1}/{len(_rv_results_in)}...")

print("\n=== Reverify stats ===")
for k, v in sorted(reverify_stats.items()):
    print(f"  {k}: {v}")

_rv_out = {
    "source_file": str(REVERIFY_INPUT_FILE),
    "reverify_scope": "human_review_only",
    "reverify_qa_statuses": sorted(REVERIFY_QA_STATUSES),
    "num_samples_total": len(_rv_results_all),
    "reverify_config": {
        "enable_verifier": ENABLE_VERIFIER,
        "verifier_model": VERIFIER_MODEL,
        "force_verify": True,
    },
    "num_samples": len(reverified_results),
    "stats": reverify_stats,
    "results": reverified_results,
}

with open(REVERIFY_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json_lib.dump(_rv_out, f, ensure_ascii=False, indent=2, default=str)

print(f"\nDa ghi: {REVERIFY_OUTPUT_FILE}")
print("MongoDB: KHONG thay doi (persist=False)")

## 8.7. Full reverify — notebook riêng (Colab)

Full reverify (~2011 mẫu) **không chạy ở đây**.

Mở trên Google Colab: **`data/code/run_full_reverify_colab.ipynb`**

Sau khi chạy xong → `hasaki_labelled_full_verified.json` trên Drive, rồi:

```bash
python scripts/build_d2_verified_dataset.py --mode full
python scripts/sample_d3_human_gold.py --n 300 --seed 42
python scripts/build_d1_d2_train.py
```

(Local: `python scripts/run_full_reverify.py`)


In [ ]:
# Full reverify: xem run_full_reverify_colab.ipynb (Colab) hoặc scripts/run_full_reverify.py (local)
print("Open: data/code/run_full_reverify_colab.ipynb")

## Ghi chú & Graph-Aware Semantic Retrieval

### ✅ Đã triển khai Graph-Aware Semantic Embedding Retrieval (Level 3)

**Tính năng mới:**

1. **Ambiguity Filtering:** Lọc câu ngắn/mơ hồ không thể annotate
   - Check độ dài: ít nhất 8 ký tự có nghĩa
   - Check từ có nghĩa: ít nhất 2 từ dài ≥3 ký tự (loại stop words)
   - Check pattern chỉ có từ hỏi: "Có không?", "Gì?", "Sao?"
   - Status: `skipped_ambiguous` với reason cụ thể

2. **Graph-Aware Intent Descriptions:** Mỗi intent node được làm giàu với thông tin ngữ cảnh:
   - Full path: `L1.L2.L3` 
   - Parent category (L2)
   - Sibling intents (cùng L2, giới hạn 5)
   - Description, detection signals, examples

3. **Sentence Embedding:** Sử dụng `paraphrase-multilingual-MiniLM-L12-v2`
   - Normalize embeddings với L2 norm
   - Lưu trữ trong MongoDB: `intent_nodes.embedding` + `graph_aware_text`

4. **Semantic Retrieval:** Cosine similarity với ngưỡng ≥ 0.3
   - Input: user sentence → embedding
   - Output: top-K intents với semantic scores

5. **Union Retrieval:** Kết hợp regex + semantic
   - Deduplication by `_id`
   - Preserve original document structure
   - Mark retrieval method (`regex` | `semantic`)

6. **Integration:** Enhanced `predict_intent()`
   - Pre-check: Skip ambiguous sentences
   - `use_semantic=True` (default): union retrieval
   - `use_semantic=False`: fallback regex-only
   - Giữ nguyên gpt-4o-mini prompt, guardrails, thresholds

**Constraints tuân thủ:**
- ✅ KHÔNG tự tạo intents mới (trừ existing auto-create logic)  
- ✅ KHÔNG loosened confidence thresholds
- ✅ KHÔNG thay đổi schema (chỉ thêm embedding fields + ambiguous_reason)
- ✅ Giữ nguyên helper function style
- ✅ KHÔNG refactor unrelated logic
- ✅ Ambiguous sentences: Skip prediction, save với qa_status="skipped_ambiguous"

### Chạy một lần để setup embeddings:
```python
embed_and_store_intent_nodes(db)  # One-time enrichment
```

### Force re-embed sau khi thêm intent mới:
```python
# Xóa embeddings cũ để force rebuild với intent mới
db.intent_nodes.update_many(
    {"level": "intent"},
    {"$unset": {"embedding": "", "graph_aware_text": ""}}
)
embed_and_store_intent_nodes(db)  # Re-embed với intent graph mới
```

### Index gợi ý:
```python 
db.intent_annotations.create_index("sample_id", unique=True)
db.intent_nodes.create_index([("level", 1), ("l3", 1)])
db.intent_nodes.create_index("embedding")  # For semantic search performance
```

### Auto-create label:
- `ALLOW_AUTO_ADD_NEW_LABEL=True` + conf ≥ `MIN_CONF_AUTO_ADD_NEW_LABEL` → upsert nhãn mới vào graph **chỉ khi pass guardrail**.

### LLM (gpt-4o-mini)

| Biến | Giá trị mặc định | Ghi chú |
|------|------------------|---------|
| `OPENAI_MODEL` | `gpt-4o-mini` | Có thể đổi `gpt-4o`, `gpt-4o-mini-2024-07-18`, ... |
| `OPENAI_API_KEY` | (getpass) | Đọc từ env `OPENAI_API_KEY`; trống thì notebook hỏi qua `getpass` (không in ra). |
| `OPENAI_TEMPERATURE` | `0` | Để xác định (deterministic) cho gán nhãn. |
| `OPENAI_MAX_RETRIES` | `3` | Backoff `2^attempt` giây cho lỗi rate-limit / mạng. |

OpenAI gọi với `response_format={"type": "json_object"}` → luôn ra JSON sạch, parser `extract_json_object` vẫn được giữ nguyên làm fallback.

Gọi API:
- §5 tạo `openai_client`; chạy lại §5 sau mỗi lần restart runtime.
- `predict_intent()` đi qua `_call_llm()` → gọi `openai_client.chat.completions.create(...)`.

### Guardrail tạo intent mới (`_validate_pred_for_auto_create`)

Trước khi upsert vào MongoDB, mọi (L1, L2, L3) phải qua các kiểm tra:

| Quy tắc | Hành vi nếu sai |
|--------|-----------------|
| **L1 phải normalize được về `truoc_mua_hang` hoặc `sau_mua_hang`** (chấp nhận `truoc_ban`, `truoc ban`, `trước bán`, ...) | `qa_status = pending_new_label_invalid_slug`, KHÔNG upsert |
| L2 phải khớp regex `^[a-z][a-z0-9_]+$` (slug ASCII lowercase + underscore) | reject |
| L2 không được nằm trong tập slug L2 hay bị nhầm thành L1 (`gia_so_sanh`, `san_pham_so_sanh`, ...) | reject |
| L3 phải khớp regex slug + ≠ L2 | reject |
| Cả 3 không rỗng | reject |

→ Hết tình trạng MongoDB sinh node `l1:gia_so_sanh`, `l1:trước bán`, `l1:my_pham`. Lý do reject lưu ở `auto_create_rejected_reason` trong `intent_annotations` để dễ rà soát:

```python
db.intent_annotations.find({"qa_status": "pending_new_label_invalid_slug"},
                            {"sample_id": 1, "intent": 1, "auto_create_rejected_reason": 1})
```

### Quy trình sau khi cập nhật `unified_intents.csv`

Mỗi lần thêm / sửa intent trong CSV (vd. INT-129…132 cluster `bao_quan_san_pham`):

1. Mở `intent_graph_rag_colab.ipynb` → chạy §5 (drop & rebuild graph từ CSV).
2. Quay lại notebook này, chạy lại §7 với `FORCE_REEMBED=1` (env hoặc set tay) để embedding khớp với taxonomy mới:
   ```python
   os.environ["FORCE_REEMBED"] = "1"  # hoặc gọi trực tiếp:
   embed_and_store_intent_nodes(db, force=True)
   ```
3. Chạy lại các batch — semantic retrieval mới sẽ trả ứng viên đúng cluster mới.

Bỏ qua bước 2 → semantic retrieval vẫn dùng embedding **cũ** (intent mới sẽ vô hình với semantic), khiến model rơi vào `pending_new_label_review` cho các câu thuộc cluster mới.
